In [290]:
# explore the gym environment
import gym
import numpy as np
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset

env = gym.make('MountainCar-v0') #action = (0,1,2) = (left, no_act, right)
print(env.observation_space)
print(env.action_space)


Box(-1.2000000476837158, 0.6000000238418579, (2,), float32)
Discrete(3)


In [291]:
observation1 = env.reset()
observation2 = env.reset()
print(observation1, observation2)

[-0.56352471  0.        ] [-0.41497223  0.        ]


In [292]:
class Q_net(nn.Module):
    def __init__(self):
        super(Q_net, self).__init__()
        self.fc1 = nn.Linear(2,16)
        self.ReLU1 = nn.LeakyReLU(inplace=True)
        self.fc2 = nn.Linear(16,24)
        self.ReLU2 = nn.LeakyReLU(inplace=True)
        self.fc3 = nn.Linear(24,3)
    def forward(self, x):
        out = self.ReLU1(self.fc1(x))
        out = self.ReLU2(self.fc2(out))
        out = self.fc3(out)
        return out

In [293]:
#Create data set
class RLDataset(Dataset):
    def __init__(self, samples, transform = None, target_transform = None):
        #samples = [(s,a,r,s_), ...]
        self.samples = self.transform(samples)
    def __getitem__(self, index):
        #if self.transform is not None:
        #    img = self.transform(img) 
        return self.samples[index]
    def __len__(self):
        return len(self.samples)
    def transform(self, samples):
        transSamples = []
        for (s,a,r,s_) in samples:
            sT = torch.tensor(s,).float()
            sT_ = torch.tensor(s_).float()
            transSamples.append((sT, a, r, sT_))
        return transSamples

In [294]:
# to modify reward, energy of the car is computed.
# height:   h = 0.45*sin(3x)+0.55
def energy(state) :
    x = state[0]
    v = state[1]
    m = 1.0
    g = 0.0025
    h = 0.45*np.sin(3*x) + 0.55
    u = m * g * h
    k = 0.5 * v * v
    xmax = 0.6
    vmax = 0.07
    n = 1 / ((m * g *(0.45*np.sin(3*xmax) + 0.55)) + (0.5 * vmax * vmax))
    e = n * (u + k)
    return e

In [295]:
# get samples: (s,a,r,s') for many episodes from env, following e-greedy
# arguments: 
#    - env: environment
#    - model: current policy
#    - episodes: the number of episodes to be generated
#    - max_steps: the max length of one episode
#    - epsilon: epsilon in e-greedy
# return: train_samples[], a list of (s,a,r,s') for (episodes) episodes
# Note: the reward is modified (otherwise, it won't converge) in 3 different ways
#       1) based on energy
#       2) based on location x only
#       3) using exponential function of x
#       One of them is used.

def GetSamplesFromEnv(env, model, episodes, max_steps=200, epsilon = 0.8):
    train_samples = []
    each_sample = None
    env.reset()
    observation_new = None  # state s
    observation_old = None  # state s'
    model.eval()
    for i_episode in range(episodes):
        #observation_new = env.reset()
        observation_old = env.reset()
        for t in range(max_steps):
            #env.render()
            #print(observation)
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                #inputT = torch.tensor(observation_new).float()
                inputT = torch.tensor(observation_old).float()
                action = torch.argmax(model(inputT)).item()
                #print(action)
            observation_new, reward, done, info = env.step(action)
            #print(reward)
            #We record samples.
            if t > 0 :
                # adjust reward based on energy
                # reference: https://colab.research.google.com/drive/1T9UGr7vdXj1HYE_4qo8KXptIwCS7S-3v
                # best results: average steps =97.3 per episode
                
                reward = (reward * 0.01) + (energy(observation_new) - energy(observation_old))
                if observation_new[0] >= 0.5 :
                    reward = reward + 1.0
                else :
                    reward = reward
                           
                # adjust reward based on its position only
                # best solution:  average steps =108
                '''
                if observation_new[0] > -0.1:
                    reward += 0.7
                elif observation_new[0] > -0.15:
                    reward += 0.5
                elif observation_new[0] > -0.2:
                    reward += 0.2
                '''
                # adjust reward based on exp function
                # best result: average steps =127
                '''
                if observation_new[0] > 0.05:
                    reward += np.exp(observation_new[0]*5)
                if done:
                    reward += 100
                '''
                #print(reward)
                each_sample = (observation_old, action, reward, observation_new)
                train_samples.append(each_sample)

            observation_old = observation_new

            if done:
            	#Failed samples are not printed
                # print successful episodes
                if t != 199:
                    print("Episode finished after {} timesteps".format(t+1))
                break
    return train_samples


In [296]:
# train the q_net, based on a fixed dataset.
# arguments
#     - target_net:  the target network, copied from q_net every 100 iterations
#     - q_net:  the Q network, updated every iteration
#     - trainloader: load the dataset for training
#     - criterion: loss function, e.g. nn.MSELoss()
#     - optimizer: optimizer, e.g., Adam
#     - device: cpu or gpu
#     - epoch_total: the number of epochs for training
#     - gamma: reward discount

def TrainNet(target_net, q_net, trainloader, criterion, optimizer, device, epoch_total, gamma):
    running_loss = 0.0
    iter_times = 0
    target_net.eval()
    q_net.train()
    for epoch in range(epoch_total):
        #print('epoch', epoch)
        running_loss = 0.0
        #if epoch == epoch_total: 
        #    break        
        for i, data in enumerate(trainloader, 0):
            # update target_net every 100 iterations
            if iter_times % 100 == 0:
                #print('updata target net')
                target_net.load_state_dict(q_net.state_dict())
                iter_times=0
            s,a,r,s_ = data
            optimizer.zero_grad()

            #output = Q_predicted.
            q_t0 = q_net(s)    # prediction of q_net
            q_t1 = target_net(s_).detach()
            target = r +gamma * torch.max(q_t1,dim=1)[0]   # target
            loss = criterion(target, torch.gather(q_t0, dim=1, index=a.unsqueeze(1)).squeeze(1))
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            iter_times += 1
    target_net.load_state_dict(q_net.state_dict())    
    #print('Finished Training')



In [305]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
q_net = Q_net()     # initial q_net
target_net = Q_net()  # initial target_net

criterion = nn.MSELoss()  # loss function
optimizer = torch.optim.Adam(q_net.parameters(),lr=0.01)   # optimmizer

train_samples = []    # define a list of (s,a,r,s')
num_datasets = 300       # number of datasets

epsilon = 1.             # initial epsilon
END_eps= num_datasets//2  # epsilon decays in the first half
START_eps=1
eps_step = epsilon / (END_eps-START_eps) # decay amount after each dataset

episodes =10     # for each dataset, update 10 new episodes
shortest_len = episodes *160  # initial the best length for 10 episodes (1600)

epochs = 12  # train for 12 epochs at each dataset

In [306]:
# training loop
# for each new dataset:
#     1) set a new epsilon
#     2) collect episodes(e.g.10) episodes by GetSamplesFromEnv()
#     3) update the dataset train_samples (limited to 4000)
#     4) trainloader the dataset
#     5) train q_net on the updated dataset

for i in range(num_datasets):
    # monitor epsilon for each dataset
    print('dataset:', i, 'epsilon', epsilon)
    # decay epsilon
    if END_eps-2 >= i >= START_eps:
        epsilon -= eps_step
    # collect new 10 episodes 
    tmpSample = GetSamplesFromEnv(env,q_net, episodes, 200, epsilon)
    
    print(len(tmpSample)) # the total number time steps for 10 episodes
    # save the model that generates the shortest steps over 10 episodes
    if len(tmpSample) < shortest_len:
        print("best model!save it!")
        torch.save(q_net.state_dict(), "bestmodel" + ".pth")
        shortest_len = len(tmpSample)
        
    # update dataset train_samples by adding tempSample
    # dataset contains the latest samples not exceeding 4000
    train_samples += tmpSample
    if len(train_samples) > 4000:
        train_samples = train_samples[len(tmpSample):len(train_samples)]
    
    trainset = RLDataset(train_samples)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=0,pin_memory=True)
    
    TrainNet(target_net, q_net, trainloader, criterion, optimizer, device, epochs, 0.9)

env.close()

dataset: 0 epsilon 1.0
1990
dataset: 1 epsilon 1.0
1990
dataset: 2 epsilon 0.9932885906040269
1990
dataset: 3 epsilon 0.9865771812080537
1990
dataset: 4 epsilon 0.9798657718120806
1990
dataset: 5 epsilon 0.9731543624161074
1990
dataset: 6 epsilon 0.9664429530201343
1990
dataset: 7 epsilon 0.9597315436241611
1990
dataset: 8 epsilon 0.953020134228188
1990
dataset: 9 epsilon 0.9463087248322148
1990
dataset: 10 epsilon 0.9395973154362417
1990
dataset: 11 epsilon 0.9328859060402686
1990
dataset: 12 epsilon 0.9261744966442954
1990
dataset: 13 epsilon 0.9194630872483223
1990
dataset: 14 epsilon 0.9127516778523491
1990
dataset: 15 epsilon 0.906040268456376
1990
dataset: 16 epsilon 0.8993288590604028
1990
dataset: 17 epsilon 0.8926174496644297
1990
dataset: 18 epsilon 0.8859060402684565
1990
dataset: 19 epsilon 0.8791946308724834
1990
dataset: 20 epsilon 0.8724832214765103
1990
dataset: 21 epsilon 0.8657718120805371
1990
dataset: 22 epsilon 0.859060402684564
1990
dataset: 23 epsilon 0.852348993

1916
dataset: 111 epsilon 0.2617449664429541
Episode finished after 197 timesteps
Episode finished after 131 timesteps
Episode finished after 132 timesteps
Episode finished after 169 timesteps
Episode finished after 159 timesteps
Episode finished after 183 timesteps
1761
dataset: 112 epsilon 0.25503355704698094
1990
dataset: 113 epsilon 0.2483221476510078
Episode finished after 199 timesteps
Episode finished after 182 timesteps
Episode finished after 118 timesteps
Episode finished after 122 timesteps
Episode finished after 137 timesteps
Episode finished after 168 timesteps
Episode finished after 117 timesteps
1633
dataset: 114 epsilon 0.24161073825503465
1990
dataset: 115 epsilon 0.2348993288590615
1990
dataset: 116 epsilon 0.22818791946308836
Episode finished after 150 timesteps
Episode finished after 153 timesteps
Episode finished after 162 timesteps
Episode finished after 153 timesteps
Episode finished after 153 timesteps
Episode finished after 160 timesteps
Episode finished after 1

Episode finished after 92 timesteps
Episode finished after 177 timesteps
Episode finished after 107 timesteps
Episode finished after 107 timesteps
Episode finished after 168 timesteps
Episode finished after 171 timesteps
1447
dataset: 146 epsilon 0.026845637583893852
Episode finished after 130 timesteps
Episode finished after 158 timesteps
Episode finished after 157 timesteps
Episode finished after 168 timesteps
1803
dataset: 147 epsilon 0.020134228187920697
Episode finished after 113 timesteps
Episode finished after 112 timesteps
Episode finished after 154 timesteps
Episode finished after 88 timesteps
Episode finished after 86 timesteps
Episode finished after 153 timesteps
Episode finished after 153 timesteps
Episode finished after 153 timesteps
Episode finished after 112 timesteps
Episode finished after 87 timesteps
1201
dataset: 148 epsilon 0.013422818791947542
Episode finished after 171 timesteps
Episode finished after 171 timesteps
Episode finished after 157 timesteps
Episode fini

Episode finished after 137 timesteps
Episode finished after 180 timesteps
Episode finished after 136 timesteps
Episode finished after 136 timesteps
1479
dataset: 185 epsilon 0.006711409395974388
1990
dataset: 186 epsilon 0.006711409395974388
1990
dataset: 187 epsilon 0.006711409395974388
1990
dataset: 188 epsilon 0.006711409395974388
Episode finished after 153 timesteps
Episode finished after 153 timesteps
Episode finished after 152 timesteps
Episode finished after 159 timesteps
Episode finished after 153 timesteps
Episode finished after 153 timesteps
Episode finished after 154 timesteps
Episode finished after 196 timesteps
Episode finished after 153 timesteps
Episode finished after 154 timesteps
1570
dataset: 189 epsilon 0.006711409395974388
1990
dataset: 190 epsilon 0.006711409395974388
1990
dataset: 191 epsilon 0.006711409395974388
1990
dataset: 192 epsilon 0.006711409395974388
1990
dataset: 193 epsilon 0.006711409395974388
1990
dataset: 194 epsilon 0.006711409395974388
1990
dataset

Episode finished after 127 timesteps
Episode finished after 174 timesteps
Episode finished after 127 timesteps
Episode finished after 135 timesteps
Episode finished after 129 timesteps
Episode finished after 136 timesteps
Episode finished after 176 timesteps
Episode finished after 127 timesteps
1447
dataset: 276 epsilon 0.006711409395974388
1990
dataset: 277 epsilon 0.006711409395974388
Episode finished after 130 timesteps
Episode finished after 131 timesteps
Episode finished after 140 timesteps
Episode finished after 131 timesteps
Episode finished after 135 timesteps
Episode finished after 132 timesteps
1589
dataset: 278 epsilon 0.006711409395974388
Episode finished after 97 timesteps
Episode finished after 163 timesteps
Episode finished after 135 timesteps
Episode finished after 122 timesteps
Episode finished after 158 timesteps
Episode finished after 199 timesteps
Episode finished after 101 timesteps
Episode finished after 161 timesteps
Episode finished after 105 timesteps
Episode f

In [307]:
test_model = Q_net()
test_model.load_state_dict(torch.load("bestmodel.pth"))
test_model.eval()
with torch.no_grad():
    for i in range (10): 
        done = False
        observation = env.reset()
        t=0
        while not done:
            t += 1
            env.render()
            inputT = torch.tensor(observation).float()
            action = torch.argmax(test_model(inputT)).item()
            
            # another policy based on location and speed regions
            # according to our intuition. 
            '''
            if observation[0] <= -0.5 and observation[1]>=0:
                action = 2
            if observation[0] <=-0.5 and observation[1]<0:
                action = 0
            if observation[0] >=-0.5 and observation[1]<=0:
                action = 0
            if observation[0] >=-0.5 and observation[1]>0:
                action = 2
            '''
            observation_new, reward, done, info = env.step(action)
            #print(action,observation_new, reward, done)
            observation = observation_new
            if done:
                if t <= 199:
                    print("success at time step", t, "episode", i)

                break
env.close()           

success at time step 109 episode 0
success at time step 109 episode 1
success at time step 110 episode 2
success at time step 112 episode 3
success at time step 111 episode 4
success at time step 109 episode 5
success at time step 110 episode 6
success at time step 109 episode 7
success at time step 111 episode 8
success at time step 110 episode 9


In [301]:
env.reset()

array([-0.59218454,  0.        ])

In [265]:
len(train_samples)

3980

In [ ]:
import numpy as np
np.shape(train_samples)